In [1]:
import re
import requests
from bs4 import BeautifulSoup
import time
import pandas as pd
from datetime import date, datetime
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import os

In [ ]:
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Accept-Language": "es-UY,es;q=0.9",
}
OPERATION_MAP = {"venta": "sell", "alquiler": "rent"}
TYPE_MAP      = {"casas": "House", "apartamentos": "Apartment"}
BASE_DOMAIN   = "https://www.infocasas.com.bo"
OUTPUT_DIR     = r"C:\Users\claud\Documents\GitHub\rent_lac\tests\scraping_output\raw"
CHECKPOINT_DIR = r"C:\Users\claud\Documents\GitHub\rent_lac\tests\scraping_output\raw\bolivia_html"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

In [3]:
# ── Exploratory cell: inspect the HTML structure of one page ──────────────
session = requests.Session()
retry = Retry(total=3, backoff_factor=2, status_forcelist=[429, 500, 502, 503, 504])
session.mount("https://", HTTPAdapter(max_retries=retry))

resp = session.get(f"{BASE_DOMAIN}/venta/apartamentos/pagina1", headers=HEADERS, timeout=20)
print("Status:", resp.status_code)
soup = BeautifulSoup(resp.text, "html.parser")

cards = soup.find_all("div", class_="lc-dataWrapper")
print(f"Cards found on page 1: {len(cards)}")

if cards:
    print("\n── First card HTML ──")
    print(cards[0].prettify())

Status: 200
Cards found on page 1: 21

── First card HTML ──
<div class="lc-dataWrapper">
 <div class="lc-owner-name">
  Melissa Nuñez
 </div>
 <a class="lc-data" href="/vendo-casa-de-3-dormitorios-en-san-antonio/193596377" target="_self" title="Vendo casa de 3 dormitorios en San Antonio">
  <div>
   <div class="lc-price" style="margin-top:30px">
    <span class="heading heading-4 high">
     <div class="property-price-tag">
      <p class="main-price">
       Gs. 430.000.000
      </p>
      <span class="commonExpenses body body-regular body-1 medium">
      </span>
     </div>
    </span>
   </div>
   <strong class="lc-location body body-1 body-bold medium">
    Casa en
    <!-- -->
    Ñemby, Central
   </strong>
  </div>
  <div class="lc-typologyContainer">
   <div class="lc-typologyTag">
    <span class="lc-typologyTag__item body body-2 body-regular medium">
     <span>
      <svg height="20" viewbox="0 0 400 400" width="20" xmlns="http://www.w3.org/2000/svg">
       <path d="M4.2

In [4]:
# ── Get total pages from the pagination element ────────────────────────────
def get_total_pages(soup):
    """Find the last page number from the pagination links."""
    # Look for pagination links like /venta/apartamentos/pagina42
    links = soup.find_all("a", href=re.compile(r"/pagina(\d+)"))
    nums = [int(re.search(r"/pagina(\d+)", a["href"]).group(1)) for a in links if a.get("href")]
    if nums:
        return max(nums)
    # Fallback: read from __NEXT_DATA__ if available
    script = soup.find("script", {"id": "__NEXT_DATA__"})
    if script:
        import json
        data = json.loads(script.string)
        try:
            return data["props"]["pageProps"]["fetchResult"]["searchFast"]["paginatorInfo"]["lastPage"]
        except (KeyError, TypeError):
            pass
    return None

print("Total pages:", get_total_pages(soup))

Total pages: 50


In [ ]:
# ── Card parser: extracts fields from a single lc-dataWrapper div ──────────
def parse_card(card, operation, property_type):
    def text(selector, attr=None, root=card):
        el = root.select_one(selector)
        if el is None:
            return None
        return el.get(attr) if attr else el.get_text(separator=" ", strip=True)

    def first_match(pattern, source):
        m = re.search(pattern, source or "")
        return m.group(1) if m else None

    full_text = card.get_text(separator=" ", strip=True)

    # ── Title ──────────────────────────────────────────────────────────────
    name = (
        text(".lc-cardName") or
        text(".lc-title") or
        text("h2") or
        text("h3")
    )

    # ── Price (prefer USD, fallback to any number) ──────────────────────────
    price_el = card.select_one(".lc-priceWrapper, .price, [class*='price']")
    price_text = price_el.get_text(strip=True) if price_el else ""
    # Remove dots/commas as thousand separators then cast
    price_raw = first_match(r"[Uu][Ss][Dd]?\s*([\d.,]+)", price_text) or first_match(r"([\d.]+)", price_text)
    try:
        price_usd = float(re.sub(r"[^\d]", "", price_raw)) if price_raw else None
    except Exception:
        price_usd = None

    # ── Numeric attributes from text: rooms / bathrooms / m2 ──────────────
    rooms      = first_match(r"(\d+)\s*(?:dorm|hab|amb)", full_text, )
    bathrooms  = first_match(r"(\d+)\s*ba[ñn]", full_text)
    m2_val     = first_match(r"(\d+(?:[.,]\d+)?)\s*m[²2]", full_text)

    # ── Location ───────────────────────────────────────────────────────────
    loc_el = card.select_one(".lc-location, .location, [class*='location'], [class*='Location']")
    location_text = loc_el.get_text(separator=",", strip=True) if loc_el else ""
    parts = [p.strip() for p in location_text.split(",") if p.strip()]
    locality = parts[0] if parts else None
    region   = parts[1] if len(parts) > 1 else None

    return {
        "name":              name,
        "price_usd":         price_usd,
        "rooms":             int(rooms)     if rooms     else None,
        "bathrooms":         int(bathrooms) if bathrooms else None,
        "m2":                float(re.sub(r",", ".", m2_val)) if m2_val else None,
        "country":           "Bolivia",
        "locality":          locality,
        "region":            region,
        "type":              TYPE_MAP[property_type],
        "operation":         OPERATION_MAP[operation],
        "consultation_date": date.today(),
    }

In [6]:
# ── Full scraping function with checkpoint support ─────────────────────────
def make_session():
    s = requests.Session()
    retry = Retry(total=5, backoff_factor=2, status_forcelist=[429, 500, 502, 503, 504])
    s.mount("https://", HTTPAdapter(max_retries=retry))
    return s


def scrape_infocasas_html(operation, property_type, max_pages=None):
    import json

    session      = make_session()
    base_url     = f"{BASE_DOMAIN}/{operation}/{property_type}"
    ckpt_csv     = os.path.join(CHECKPOINT_DIR, f"checkpoint_{operation}_{property_type}.csv")
    ckpt_meta    = os.path.join(CHECKPOINT_DIR, f"checkpoint_{operation}_{property_type}.json")

    # Resume from checkpoint if available
    if os.path.exists(ckpt_csv) and os.path.exists(ckpt_meta):
        with open(ckpt_meta) as f:
            meta = json.load(f)
        start_page  = meta["last_page"] + 1
        total_pages = meta["total_pages"]
        all_data    = pd.read_csv(ckpt_csv).to_dict("records")
        print(f"Retomando desde página {start_page}/{total_pages} ({len(all_data)} propiedades ya guardadas)")
    else:
        # Fetch page 1 to discover total pages
        first = session.get(f"{base_url}/pagina1", headers=HEADERS, timeout=20)
        first_soup  = BeautifulSoup(first.text, "html.parser")
        total_pages = get_total_pages(first_soup)
        if total_pages is None:
            print("No se pudo determinar el total de páginas — se iterará hasta que no haya resultados.")
            total_pages = 9999
        start_page = 1
        all_data   = []
        print(f"Iniciando desde página 1/{total_pages}")

    # Cap iteration if max_pages is set
    end_page = total_pages if max_pages is None else min(start_page + max_pages - 1, total_pages)

    consecutive_empty = 0

    for page in range(start_page, end_page + 1):
        url = f"{base_url}/pagina{page}"
        try:
            response = session.get(url, headers=HEADERS, timeout=20)
            soup     = BeautifulSoup(response.text, "html.parser")
            cards    = soup.find_all("div", class_="lc-dataWrapper")

            if not cards:
                consecutive_empty += 1
                print(f"Página {page}: sin cards ({consecutive_empty} vacías consecutivas)")
                if consecutive_empty >= 3:
                    print("3 páginas vacías seguidas — se asume fin del listado.")
                    break
                time.sleep(3)
                continue

            consecutive_empty = 0
            rows = [parse_card(c, operation, property_type) for c in cards]
            all_data.extend(rows)

            # Save checkpoint (only when doing a full run)
            if max_pages is None:
                pd.DataFrame(all_data).to_csv(ckpt_csv, index=False, encoding="utf-8-sig")
                with open(ckpt_meta, "w") as f:
                    json.dump({"last_page": page, "total_pages": total_pages}, f)

            print(f"Página {page}/{total_pages} — {len(rows)} propiedades (total: {len(all_data)})")
            time.sleep(2)

        except Exception as e:
            print(f"Error en página {page}: {e} — reintentando en 10s...")
            time.sleep(10)
            continue

    # Clean up checkpoints
    if max_pages is None:
        for f in [ckpt_csv, ckpt_meta]:
            if os.path.exists(f):
                os.remove(f)

    return pd.DataFrame(all_data)

In [7]:
# ── Run for all combinations and save ─────────────────────────────────────
dfs = []

for operation in OPERATION_MAP:
    for property_type in TYPE_MAP:
        print(f"\n=== {operation} / {property_type} ===")
        df = scrape_infocasas_html(operation, property_type)
        dfs.append(df)

df_final = pd.concat(dfs, ignore_index=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
filename  = os.path.join(OUTPUT_DIR, f"bronze_full_{timestamp}.csv")
df_final.to_csv(filename, index=False, encoding="utf-8-sig")

print(f"\nTotal: {len(df_final)} propiedades")
print(f"Guardado: {filename}")
df_final.head()


=== venta / casas ===
Iniciando desde página 1/50
Página 1/50 — 21 propiedades (total: 21)
Página 2/50 — 21 propiedades (total: 42)
Página 3/50 — 21 propiedades (total: 63)
Página 4/50 — 21 propiedades (total: 84)
Página 5/50 — 21 propiedades (total: 105)
Página 6/50 — 21 propiedades (total: 126)
Página 7/50 — 21 propiedades (total: 147)
Página 8/50 — 21 propiedades (total: 168)
Página 9/50 — 21 propiedades (total: 189)
Página 10/50 — 21 propiedades (total: 210)
Página 11/50 — 21 propiedades (total: 231)
Página 12/50 — 21 propiedades (total: 252)
Página 13/50 — 21 propiedades (total: 273)
Página 14/50 — 21 propiedades (total: 294)
Página 15/50 — 21 propiedades (total: 315)
Página 16/50 — 21 propiedades (total: 336)
Página 17/50 — 21 propiedades (total: 357)
Página 18/50 — 21 propiedades (total: 378)
Página 19/50 — 21 propiedades (total: 399)
Página 20/50 — 21 propiedades (total: 420)
Página 21/50 — 21 propiedades (total: 441)
Página 22/50 — 21 propiedades (total: 462)
Página 23/50 — 2

,name,price_usd,rooms,bathrooms,m2,country,locality,region,type,operation,consultation_date
0,Vendo casa en b° nazareth,NaN,3.0,2.0,787.0,Paraguay,Casa en,Nazareth,House,sell,2026-04-28
1,En venta lujosa residencia en lambare zona del...,750000.0,NaN,NaN,840.0,Paraguay,Casa en,Lambaré,House,sell,2026-04-28
2,Casa con consultorio en villa aurelia,NaN,4.0,3.0,200.0,Paraguay,Casa en,Villa Aurelia,House,sell,2026-04-28
3,Residencia en esquina sobre brasilia a metros ...,1500000.0,NaN,NaN,800.0,Paraguay,Casa en,Mcal. López,House,sell,2026-04-28
4,Vendo duplex a estrenar en ñemby,NaN,NaN,NaN,1.0,Paraguay,Casa en,Ñemby,House,sell,2026-04-28
